# Step 3: Brain node (neuron morphologies)

Same logic as the universe node: calibrate first on labeled sets, then run real data, with predictions written beforehand.

**Predictions (written before any real neuron is run):**
1. Real dendritic arbors give a correlation dimension between 1.2 and 1.8 over the window, with local slope rising as r grows toward a single branch-length scale. (Published box-counting values for single neurons are usually reported near 1.3-1.7; I'm not certain of the exact range.)
2. The fixed-segment control (one length scale) is rejected by slope drift.
3. D estimated at N and at N/2 agrees within 0.15 for real arbors. If not, N is too small and the number is not usable.

Real data only: this notebook does NOT generate fake neurons. Upload SWC files (e.g. from NeuroMorpho.org) to /content/swc/.

In [ ]:
import glob, os
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree

rng = np.random.default_rng(42)
N = 3000                      # points per arbor, same for every dataset (matched N)
WIN = (0.02, 0.20)            # window as fraction of extent; lower end raised to 5x point spacing if needed
SWC_DIR = "/content/swc"
SWC_TYPES = {3, 4}            # 3 = basal dendrite, 4 = apical dendrite; add 2 for axon, 1 for soma

## Estimator (identical for every dataset)

In [ ]:
def corr_curve(p, spacing=None, n_r=16):
    ext = (p.max(0) - p.min(0)).max()
    lo = WIN[0] * ext
    if spacing is not None:
        lo = max(lo, 5 * spacing)          # stay well above the point spacing along the path
    hi = WIN[1] * ext
    r = np.logspace(np.log10(lo), np.log10(hi), n_r)
    t = cKDTree(p); n = len(p)
    c = (t.count_neighbors(t, r) - n) / (n * (n - 1))
    ok = c > 0
    return r[ok], c[ok]

def local_slopes(r, c, w=4):
    lr, lc = np.log(r), np.log(c)
    return np.array([np.polyfit(lr[i:i + w], lc[i:i + w], 1)[0] for i in range(len(lr) - w + 1)])

def analyse(p, spacing=None):
    r, c = corr_curve(p, spacing)
    D = np.polyfit(np.log(r), np.log(c), 1)[0]
    sl = local_slopes(r, c)
    return D, sl, np.log10(r[-1] / r[0])

def conv_check(p, path_len=None):
    """D at N versus D at N/2 (random half). Large gap = N too small."""
    sp = None if path_len is None else path_len / len(p)
    d_full = analyse(p, sp)[0]
    half = p[rng.choice(len(p), len(p) // 2, replace=False)]
    sp_h = None if path_len is None else path_len / len(half)
    return d_full, analyse(half, sp_h)[0]

## Labeled synthetic sets (calibration)
Trees use branch length ratio r per level with two children, so the similarity dimension is log2/log(1/r).
That value is a reference (an upper bound when branches overlap), not exact ground truth.

In [ ]:
def resample_segments(a, b, ds):
    """Evenly spaced points along segments a->b (arrays of shape (m,3))."""
    L = np.linalg.norm(b - a, axis=1)
    k = np.maximum((L / ds).astype(int), 1)
    pts = []
    for ai, bi, ki in zip(a, b, k):
        t = (np.arange(ki) + 0.5) / ki
        pts.append(ai + np.outer(t, bi - ai))
    return np.vstack(pts), L.sum()

def rot(v, axis, ang):
    axis = axis / np.linalg.norm(axis)
    return v * np.cos(ang) + np.cross(axis, v) * np.sin(ang) + axis * np.dot(axis, v) * (1 - np.cos(ang))

def binary_tree(ratio, depth=11, theta=np.radians(38)):
    a_list, b_list = [], []
    stack = [(np.zeros(3), np.array([0, 0, 1.0]), 1.0, 0)]
    while stack:
        o, d, l, lev = stack.pop()
        e = o + d * l
        a_list.append(o); b_list.append(e)
        if lev < depth:
            ax = np.cross(d, rng.normal(size=3))
            for s in (+1, -1):
                stack.append((e, rot(d, ax, s * theta), l * ratio, lev + 1))
    a, b = np.array(a_list), np.array(b_list)
    L = np.linalg.norm(b - a, axis=1).sum()
    pts, _ = resample_segments(a, b, L / N)
    return pts, L

def fixed_segments(n_seg=300, seg_len=0.04):
    o = rng.random((n_seg, 3)); d = rng.normal(size=(n_seg, 3)); d /= np.linalg.norm(d, axis=1, keepdims=True)
    a, b = o, o + d * seg_len
    L = seg_len * n_seg
    pts, _ = resample_segments(a, b, L / N)
    return pts, L

def line():
    p = np.zeros((N, 3)); p[:, 0] = np.linspace(0, 1, N); return p, 1.0

def sheet():
    p = np.zeros((N, 3)); p[:, :2] = rng.random((N, 2)); return p, None

def brownian():
    return np.cumsum(rng.normal(size=(N, 3)), axis=0), None

def poisson():
    return rng.random((N, 3)), None

def to_N(p):
    return p[rng.choice(len(p), N, replace=False)] if len(p) > N else p

tests = {
    "Line (D=1)": (line, 1.0),
    "Brownian path (D=2)": (brownian, 2.0),
    "Sheet in 3D (D=2)": (sheet, 2.0),
    "Tree ratio 0.60": (lambda: binary_tree(0.60), np.log(2) / np.log(1 / 0.60)),
    "Tree ratio 0.65": (lambda: binary_tree(0.65), np.log(2) / np.log(1 / 0.65)),
    "Tree ratio 0.70": (lambda: binary_tree(0.70), np.log(2) / np.log(1 / 0.70)),
    "Poisson cloud (D=3)": (poisson, 3.0),
    "Fixed-length segments, NOT fractal": (fixed_segments, None),
}

curves, rows = {}, []
for name, (gen, ref) in tests.items():
    p, L = gen()
    p = to_N(p)
    sp = None if L is None else L / len(p)
    D, sl, dec = analyse(p, sp)
    d_full, d_half = conv_check(p, L)
    curves[name] = sl
    rows.append((name, ref, D, sl[0], sl[-1], dec, abs(d_full - d_half)))
    print(f"{name:36s} ref={'--' if ref is None else f'{ref:.2f}'}  D={D:.3f}  local {sl[0]:.2f}->{sl[-1]:.2f}  decades={dec:.2f}  |D(N)-D(N/2)|={abs(d_full - d_half):.3f}")

## Real neurons
Upload SWC files to /content/swc/ (Colab left sidebar, folder icon). Each file is one neuron. SWC nodes are unevenly spaced,
so each parent-child edge is resampled evenly along its path before anything is measured.

In [ ]:
def load_swc(path, types=SWC_TYPES):
    d = np.loadtxt(path, comments="#")
    ids, typ, xyz, par = d[:, 0].astype(int), d[:, 1].astype(int), d[:, 2:5], d[:, 6].astype(int)
    idx = {i: k for k, i in enumerate(ids)}
    a, b = [], []
    for k in range(len(ids)):
        if par[k] in idx and typ[k] in types:
            a.append(xyz[idx[par[k]]]); b.append(xyz[k])
    a, b = np.array(a), np.array(b)
    L = np.linalg.norm(b - a, axis=1).sum()
    pts, _ = resample_segments(a, b, L / (N * 1.5))   # oversample slightly, then draw exactly N
    return to_N(pts), L

files = sorted(glob.glob(os.path.join(SWC_DIR, "*.swc")))
if not files:
    print(f"No SWC files in {SWC_DIR}. Upload some and re-run this cell (no fake data is generated).")
real = {}
for f in files:
    try:
        p, L = load_swc(f)
        if len(p) < N:
            print(f"{os.path.basename(f)}: only {len(p)} points after resampling, skipped (needs >= {N})"); continue
        D, sl, dec = analyse(p, L / len(p))
        d_full, d_half = conv_check(p, L)
        real[os.path.basename(f)] = sl
        print(f"{os.path.basename(f):28s} D={D:.3f}  local {sl[0]:.2f}->{sl[-1]:.2f}  decades={dec:.2f}  |D(N)-D(N/2)|={abs(d_full - d_half):.3f}  path_length={L:.0f}")
    except Exception as e:
        print(f"{os.path.basename(f)}: failed ({e})")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))
for name, sl in curves.items():
    ax.plot(np.linspace(0, 1, len(sl)), sl, "--", alpha=0.6, label=name)
for name, sl in real.items():
    ax.plot(np.linspace(0, 1, len(sl)), sl, "-", lw=2, label="REAL " + name)
ax.set_xlabel("position in window (small r to large r)"); ax.set_ylabel("local correlation dimension")
ax.legend(fontsize=6); ax.set_title("Brain node: local dimension vs scale"); plt.tight_layout()
plt.savefig("brain_node.png", dpi=150); plt.show()

## Reading the result
- Score against the three predictions at the top, whatever the outcome.
- A neuron whose D changes by more than 0.15 between N and N/2 is under-sampled; report it as such.
- For the trifold comparison we line up D(r) curves (universe, brain, AI), not single numbers, and always at matched N with controls.